<a href="https://colab.research.google.com/github/MRoy03/Forensic_LAB/blob/main/notebooks/00_colab_setup_and_drive_mount.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 -- Colab/Kaggle Setup & Drive Mount

Run this first in a fresh Colab/Kaggle session. It:
1. Clones the project from GitHub (if not already present in this session).
2. Mounts Google Drive (for large datasets: DFDC, FaceForensics++).
3. Installs `requirements-train.txt` (the PyTorch stack -- never needed
   for the served app itself, only for training/export).

**Dataset layout expected on Drive** (see `config/settings.yaml` ->
`data.root_drive` and `data.datasets_registry`):
```
/content/drive/MyDrive/ForensicAI_Lab/data/
    casia2/ comofod/ coverage/ imd2020/ ffpp_images/ dfdc_subset/ ffpp_videos/
    images/ masks/ labels.csv
```
If your datasets are sitting directly under `MyDrive/ForensicAI_Lab/` (no
`data/` subfolder) and/or still zipped, see
`00b_prepare_local_datasets.ipynb`'s "Option C" cell, which extracts them
into the right place automatically.

In [1]:
import sys, os

def _is_colab():
    try:
        import google.colab  # noqa
        return True
    except ImportError:
        return False

IS_COLAB = _is_colab()
IS_KAGGLE = (not IS_COLAB) and (os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ)
PROJECT_ROOT = "/kaggle/working/ForensicAI_Lab" if IS_KAGGLE else "/content/ForensicAI_Lab"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/MRoy03/Forensic_LAB.git {PROJECT_ROOT}

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
elif IS_KAGGLE:
    !pip install -q -r requirements-train.txt
else:
    print("Not in Colab/Kaggle -- assuming requirements-train.txt is already installed locally.")

Cloning into '/content/ForensicAI_Lab'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 120 (delta 37), reused 106 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (120/120), 73.71 KiB | 6.70 MiB/s, done.
Resolving deltas: 100% (37/37), done.
Mounted at /content/drive


In [2]:
from config.loader import get_settings, resolve_dataset_root

settings = get_settings()
dataset_root = resolve_dataset_root(settings)
print("Using dataset root:", dataset_root)

for entry in settings.data.datasets_registry:
    subdir = dataset_root / entry.local_subdir
    status = "found" if subdir.exists() else "MISSING (download/place it here)"
    print(f"  {entry.name:25s} ({entry.type:5s}) -> {subdir}  [{status}]")

Using dataset root: /content/ForensicAI_Lab/data
  CASIA2.0                  (image) -> /content/ForensicAI_Lab/data/casia2  [MISSING (download/place it here)]
  CoMoFoD                   (image) -> /content/ForensicAI_Lab/data/comofod  [MISSING (download/place it here)]
  Coverage                  (image) -> /content/ForensicAI_Lab/data/coverage  [MISSING (download/place it here)]
  IMD2020                   (image) -> /content/ForensicAI_Lab/data/imd2020  [MISSING (download/place it here)]
  FaceForensics++_images    (image) -> /content/ForensicAI_Lab/data/ffpp_images  [MISSING (download/place it here)]
  DFDC_subset               (video) -> /content/ForensicAI_Lab/data/dfdc_subset  [MISSING (download/place it here)]
  FaceForensics++_videos    (video) -> /content/ForensicAI_Lab/data/ffpp_videos  [MISSING (download/place it here)]


In [3]:
from pathlib import Path

drive_zip_dir = Path("/content/drive/MyDrive/ForensicAI_Lab")
print(f"Zip files found directly under {drive_zip_dir}:")
for z in sorted(drive_zip_dir.glob("*.zip")):
    print(f"  {z.name}  ({z.stat().st_size / 1e9:.2f} GB)")

Zip files found directly under /content/drive/MyDrive/ForensicAI_Lab:
  casia2.zip  (2.75 GB)
  comofod.zip  (3.21 GB)
  coverage.zip  (0.11 GB)
  ffpp_video.zip  (17.89 GB)
  imd2020.zip  (0.59 GB)


In [5]:
import zipfile

# Map each zip's EXACT filename (from the listing above) to the registry
# subdir it corresponds to. Edit the left-hand side to match your files.
ZIP_TO_SUBDIR = {
    "casia2.zip": "casia2",           # <- change to your actual filename
    "comofod.zip": "comofod",
    "coverage.zip": "coverage",
    "imd2020.zip": "imd2020",
    "ffpp_video": "ffpp_videos",
}

local_data_dir = Path(PROJECT_ROOT) / "data"
local_data_dir.mkdir(parents=True, exist_ok=True)

for zip_name, subdir in ZIP_TO_SUBDIR.items():
    zip_path = drive_zip_dir / zip_name
    target = local_data_dir / subdir
    if not zip_path.exists():
        print(f"[SKIP] {zip_path} not found -- check the filename")
        continue
    if target.exists() and any(target.iterdir()):
        print(f"[SKIP] {target} already populated")
        continue
    target.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {zip_path.name} -> {target} ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(target)
    print("  done")

# Re-resolve now that local data/ has content -- this should now report
# the local path (since it's a non-empty folder that exists).
dataset_root = resolve_dataset_root(settings)
print("\nUsing dataset root:", dataset_root)

[SKIP] /content/ForensicAI_Lab/data/casia2 already populated
[SKIP] /content/ForensicAI_Lab/data/comofod already populated
[SKIP] /content/ForensicAI_Lab/data/coverage already populated
[SKIP] /content/ForensicAI_Lab/data/imd2020 already populated
[SKIP] /content/drive/MyDrive/ForensicAI_Lab/ffpp_video not found -- check the filename

Using dataset root: /content/ForensicAI_Lab/data


In [6]:
import shutil

zip_name = "ffpp_video.zip"  # <- copy exactly from the "Zip files found" listing above
subdir = "ffpp_videos"

zip_path = drive_zip_dir / zip_name
target = local_data_dir / subdir

if not zip_path.exists():
    print(f"[STILL NOT FOUND] {zip_path} -- double-check the filename against the listing cell")
else:
    if target.exists():
        shutil.rmtree(target)  # clear out any stale/partial previous attempt
    target.mkdir(parents=True, exist_ok=True)

    print(f"Extracting {zip_path.name} -> {target} ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(target)

    file_count = len(list(target.rglob("*")))
    print(f"Done. {file_count} files/folders now under {target}")


Extracting ffpp_video.zip -> /content/ForensicAI_Lab/data/ffpp_videos ...
Done. 7019 files/folders now under /content/ForensicAI_Lab/data/ffpp_videos
